# Procedimento para calcular população de bacias hidrossanitárias
##### Este documento define as etapas para a obtenção do número de habitantes inseridos na Área de Prestação de Serviços de bacias de esgotamento

Importação de bibliotecas:

In [4]:
import pandas as pd
import geopandas as gpd
import os

## 1º Passo: Importação dos dados
##### Setores Censitários: https://www.ibge.gov.br/estatisticas/sociais/trabalho/22827-censo-demografico-2022.html?edicao=41852&t=resultados 
Fazer download da malha de setores centiários por UF
##### Domicílios: https://www.ibge.gov.br/estatisticas/sociais/populacao/38734-cadastro-nacional-de-enderecos-para-fins-estatisticos.html?edicao=38891&t=resultados
Selecionar arquivos por município
##### APS encaminhado pela CORSAN; Bacias delimitadas pelo Analista. Coluna com o nome das bacias também deve ser especificado.

*Buscar código do município em https://www.ibge.gov.br/explica/codigos-dos-municipios.php

In [7]:
#Caçapava do Sul
setores = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Concórdia\Cálculo de população\SC_setores_CD2022.gpkg')
domicilios = pd.read_csv(r'C:\Users\gabriel.coimbra\Desktop\Concórdia\Cálculo de população\4204301.csv',delimiter = ';')
aps = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Concórdia\Cálculo de população\Terrenos Isolados para Cami\3terrenosJuntos.gpkg')
bacias = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Concórdia\Cálculo de população\Terrenos Isolados para Cami\3terrenosJuntos.gpkg')
coluna_nome_bacias = 'Name'
caminho_exportacao = r'C:\Users\gabriel.coimbra\Desktop\Concórdia\Cálculo de população\Terrenos Isolados para Cami\popdom_bacia_2022_novo.xlsx'
crs = "EPSG:31982"

## Funções auxiliares

In [8]:
#Funções auxiliares

#contagem de domicílios em cada setor na APS

import geopandas as gpd

def somar_extensao_polig(lines_gdf, polys_gdf, poly_id_col="Nome"):
    """
    Retorna um DataFrame com o comprimento (m e km) das LINHAS contidas em cada polígono.
    Requer ambos em CRS projetado (unidades em METROS).
    - Filtra geometrias nulas/vazias
    - Valida polígonos (make_valid/buffer(0))
    - Faz overlay com keep_geom_type=False e filtra só linhas
    - Soma por polígono e adiciona coluna em km
    """
    # Cópias e colunas necessárias
    lines = lines_gdf[["geometry"]].copy()
    polys = polys_gdf[[poly_id_col, "geometry"]].copy()

    # CRS: alinhar se necessário
    if lines.crs != polys.crs:
        polys = polys.to_crs(lines.crs)

    # Limpeza: remover nulos/vazios
    lines = lines[lines.geometry.notnull() & ~lines.geometry.is_empty]
    polys = polys[polys.geometry.notnull() & ~polys.geometry.is_empty]

    # Validar polígonos (Shapely 2 -> make_valid; fallback buffer(0))
    try:
        polys["geometry"] = polys.geometry.make_valid()
    except Exception:
        polys["geometry"] = polys.buffer(0)

    # Overlay SEM restringir tipo de geometria
    inter = gpd.overlay(lines, polys, how="intersection", keep_geom_type=False)

    # Ficar só com partes lineares (descarta GeometryCollection/Polígonos/Pontos)
    inter = inter[inter.geom_type.isin(["LineString", "MultiLineString"])].copy()

    if inter.empty:
        # Retorno “vazio” com colunas esperadas
        return gpd.GeoDataFrame(
            {poly_id_col: [], "Extensão de Rede (m)": [], "Extensão de Rede (km)": []}
        )

    # Comprimento em metros (CRS deve estar em metros!)
    inter["Extensão de Rede (m)"] = inter.geometry.length

    # Soma por polígono
    out = (
        inter.groupby(poly_id_col, as_index=False)["Extensão de Rede (m)"]
        .sum()
        .sort_values(poly_id_col)
    )
    out["Extensão de Rede (km)"] = out["Extensão de Rede (m)"] / 1000.0
    return out

    return out

def contar_pontos_poligono(polygons, points, polygon_id_col="poly_id", predicate="intersects"):
    if polygons.crs != points.crs:
        points = points.to_crs(polygons.crs)

    # Garante coluna de ID
    if polygon_id_col not in polygons.columns:
        polygons = polygons.reset_index(drop=False).rename(columns={"index": polygon_id_col})

    joined = gpd.sjoin(points, polygons[[polygon_id_col, "geometry"]], predicate=predicate)
    counts = joined.groupby(polygon_id_col).size().rename("n_pontos").reset_index()
    
    result = polygons.merge(counts, on=polygon_id_col, how="left")
    result["n_pontos"] = result["n_pontos"].fillna(0).astype(int)
    
    return result

# Não é necessário mexer nisso abaixo

## 2º Passo: Tratamento dos dados

Conforme Diretriz Corsan (2025), o IBGE considera, para a densidade domiciliar, somente os domicílios particulares ocupados (v0007), o qual não representa a realidade das economias residenciais no cadastro da Corsan/Aegea. Portanto, deve-se recalcular a densidade domiciliar dos setores censitários. Para recalcular a densidade domiciliar, deve-se dividir a população (v0001) pelo total de domicílios particulares (v0003), gerando uma nova coluna “Densidade”.

In [9]:
setores['Densidade'] = setores['v0001']/setores['v0003']

É necessário filtrar os domicílios particulares (COD_ESPECIE = 1) e igrejas (COD_ESPECIE = 8) e transformar csv de domicílios em um arquivo georreferenciado

In [23]:
domparticular = domicilios[(domicilios['COD_ESPECIE'] == 1) ]
domparticular = gpd.GeoDataFrame(domparticular, geometry=gpd.points_from_xy(domparticular.LONGITUDE, domparticular.LATITUDE), crs="EPSG:4326")

Deve-se colocar tudo no mesmo CRS definido

In [24]:
domparticular = domparticular.to_crs(crs)
bacias = bacias.to_crs(crs)
aps = aps.to_crs(crs)
setores = setores.to_crs(crs) #convertendo pra sistema de coordenadas padrão

##### Intersecção entre setores e APS

In [25]:
setores_aps = gpd.clip(setores, aps) #interseção entre setores e APS
setores_aps = setores_aps[setores_aps['v0003']>0]
domparticular_aps = gpd.clip(domparticular, aps) #interseção entre domicílios e APS

## 3º Passo: Cálculo da população na APS
##### As etapas realizadas são:
- Intersecção entre Setores e APS
- Intersecção entre Domicílios e APS
- Contagem de domicílios em cada setor na APS
- Calculo da população

##### Contagem de domicílios em cada setor na APS

In [26]:
populacao_aps = contar_pontos_poligono(setores_aps, domparticular_aps)

In [27]:
populacao_aps

,poly_id,CD_SETOR,SITUACAO,CD_SIT,CD_TIPO,AREA_KM2,CD_REGIAO,NM_REGIAO,CD_UF,NM_UF,...,v0001,v0002,v0003,v0004,v0005,v0006,v0007,geometry,Densidade,n_pontos
0,3913,420430105000085,Rural,8,0,23.260148,4,Sul,42,Santa Catarina,...,519,238,234,4,2.7,0.0106,188,"POLYGON Z ((398802.258 6985409.704 0, 398976.7...",2.217949,0
1,4002,420430105000229,Urbana,1,0,0.137485,4,Sul,42,Santa Catarina,...,608,242,242,0,2.7,0.0263,228,"POLYGON Z ((398810.892 6985660.505 0, 398794.0...",2.512397,243
2,3999,420430105000224,Urbana,1,0,0.674356,4,Sul,42,Santa Catarina,...,786,355,355,0,2.6,0.0590,305,"POLYGON Z ((398976.743 6985266.526 0, 399004.1...",2.214085,324
3,3984,420430105000205,Urbana,1,0,0.329445,4,Sul,42,Santa Catarina,...,599,248,248,0,2.6,0.0216,232,"POLYGON Z ((398897.044 6985812.036 0, 398915.5...",2.415323,23
4,3981,420430105000202,Urbana,1,0,0.315392,4,Sul,42,Santa Catarina,...,948,368,366,2,2.8,0.0029,342,"POLYGON Z ((399745.532 6986993.725 0, 399745.9...",2.590164,199
5,3960,420430105000163,Urbana,1,0,0.231317,4,Sul,42,Santa Catarina,...,273,105,105,0,2.7,0.0396,101,"POLYGON Z ((399697.647 6986932.342 0, 399690.0...",2.600000,28
6,3926,420430105000101,Urbana,1,0,0.113228,4,Sul,42,Santa Catarina,...,596,287,287,0,2.4,0.0567,247,"POLYGON Z ((399634.407 6987329.043 0, 399642.6...",2.076655,275
7,3852,420430105000013,Urbana,1,0,0.158851,4,Sul,42,Santa Catarina,...,499,234,233,1,2.5,0.0051,196,"POLYGON Z ((399358.273 6987752.578 0, 399362.3...",2.141631,116
8,3851,420430105000012,Urbana,1,0,0.076463,4,Sul,42,Santa Catarina,...,536,223,223,0,2.7,0.0248,202,"POLYGON Z ((399346.867 6987746.259 0, 399333.4...",2.403587,63
9,3904,420430105000075,Urbana,1,0,0.212728,4,Sul,42,Santa Catarina,...,804,334,334,0,2.6,0.0254,315,"POLYGON Z ((399734.242 6987346.546 0, 399617.8...",2.407186,76


##### Cálculo da população com a densidade e n_pontos criado

In [29]:
populacao_aps['População 2022'] = populacao_aps['Densidade']*populacao_aps['n_pontos']

pop = populacao_aps['População 2022'].sum()
print(f"A população total na APS em 2022 é de {pop:.0f}")
econ = len(domparticular_aps)
print(f"A população total na APS em 2022 é de {econ:.0f}")

A população total na APS em 2022 é de 3813
A população total na APS em 2022 é de 1676


## 4º Passo: Cálculo da população por bacia (2022)

Após delimitar as bacias para pelo menos 90% dos domicílios do IBGE (Censo 2022), deverá ser identificado quantos domicílios estão inseridos em cada bacia. As etapas realizadas são:
- Criação de camada com domicílios classificados por setor e bacia
- Criação de camada com bacias divididas em setores
- Calculo da população com base na densidade de cada domicílio dentro de cada setor dividido pela bacia
- Agrupamento dos valores por bacia, gerando a quantidade de população e domicílios por bacia

##### Intersecções entre domicílios, setores e bacias

In [30]:
camada_unida = gpd.sjoin(
    domparticular_aps,
    bacias, 
    predicate="intersects",
    how="left"
)
camada_unida = camada_unida.drop(columns=['index_right'], errors='ignore')

dompart_setores = gpd.sjoin(
    camada_unida,
    populacao_aps,  
    predicate="intersects",
    how="left"
)

bacias_setores = gpd.sjoin(
    populacao_aps,
    bacias,  
    predicate="intersects",
    how="left"
)

#dompart_setores_filtrado = dompart_setores[[coluna_nome_bacias, 'CD_SETOR']]
dompart_setores_filtrado = dompart_setores
#bacias_setores_filtrado = bacias_setores[[coluna_nome_bacias, 'CD_SETOR','Densidade']]
bacias_setores_filtrado = bacias_setores

In [31]:
bacias_setores_filtrado.to_excel(os.path.join(r'C:\Users\gabriel.coimbra\Desktop\CORSAN\Caçapava do Sul\bacias_setores_filtrado.xlsx'))
dompart_setores_filtrado.to_excel(os.path.join(r'C:\Users\gabriel.coimbra\Desktop\CORSAN\Caçapava do Sul\dompart_setores_filtrado.xlsx'))

##### Contagem da quantidade de vezes que uma combinação Setores Censitários + Bacia aparece

In [32]:
# Passo 1: Contar ocorrências de Nome + CD_SETOR na planilha de referência
contagem = (
    dompart_setores_filtrado
    .groupby([coluna_nome_bacias, 'CD_SETOR'])
    .size()
    .reset_index(name='Domicílios')
)

# Passo 2: Fazer merge com o DataFrame base
bacias_setores_filtrado = bacias_setores_filtrado.merge(contagem, on=[coluna_nome_bacias, 'CD_SETOR'], how='left')

# Passo 3: Substituir NaN por 0 (caso não tenha ocorrência)
bacias_setores_filtrado['Domicílios'] = bacias_setores_filtrado['Domicílios'].fillna(0).astype(int)

##### Cálculo da população por combinação Setores Censitários + Bacia

In [33]:
bacias_setores_filtrado['População'] = bacias_setores_filtrado['Domicílios']*bacias_setores_filtrado['Densidade']

##### Soma da população calculada por bacia

In [34]:
bacias_populacao = bacias_setores_filtrado[[coluna_nome_bacias,'Domicílios','População']].groupby(coluna_nome_bacias).sum()
bacias_populacao

,Domicílios,População
Name,,
Area 01 - Implantação rede - 2026 - Subbaca 09 (parcial),787,1802.330960
Area 02 - Implantação rede - 2026 - SubBacia 18,590,1383.428196
Área - Rua Oswaldo Zandavalli - 2026,298,626.770865


##### Exportar excel final

In [35]:
bacias_populacao.to_excel(caminho_exportacao)

# Resultados

In [ ]:
pop_aps = populacao_aps['População 2022'].sum()
print(f"A população total na APS em 2022 é de {pop_aps:.0f}")
dom_aps = len(domparticular_aps)
print(f"Os domicílios totais na APS em 2022 é de {dom_aps:.0f}")

resultado_dompop = bacias_populacao.copy()

resultado_dompop['Dom % bacias'] = resultado_dompop['Domicílios']/(resultado_dompop['Domicílios'].sum())
resultado_dompop['Dom % APS'] = resultado_dompop['Domicílios']/dom_aps
resultado_dompop['Pop % bacias'] = resultado_dompop['População']/(resultado_dompop['População'].sum())
resultado_dompop['Pop % APS'] = resultado_dompop['População']/pop_aps
resultado_dompop['Dom % APS']
display(resultado_dompop)

In [ ]:
tem_agr = int(resultado_dompop['Dom % APS'].sum()*dom_aps)
dom_aps
novdom_aps = int(dom_aps*0.9)
posso_tirar = int(tem_agr - novdom_aps)
print(f'O total é {dom_aps}, agr tenho {tem_agr}. 90% seria {novdom_aps}. Posso tirar {posso_tirar}')

# Extensão e Área das Bacias

In [ ]:
eixolog = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\CORSAN\Caçapava do Sul\copia eixolog aumentado.gpkg')
eixolog =eixolog.to_crs(crs)
bacias_area = bacias.to_crs(crs)

bacias_area_len = somar_extensao_polig(eixolog, bacias_area, poly_id_col="Name")
bacias_area_len["Área (km²)"] = bacias_area.geometry.area / 10**6

bacias_area_len = bacias_area_len.set_index("Name")

resultado_final = resultado_dompop.merge(
    bacias_area_len,
    left_index=True,
    right_index=True,
    how="left"
)

resultado_final = resultado_final[['Domicílios','Dom % bacias','Dom % APS','População','Pop % bacias','Pop % APS','Extensão de Rede (m)','Área (km²)']].transpose()

display(resultado_final)

In [ ]:
resultado_final.to_excel(os.path.join(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Caçapava do Sul\inputpredim.xlsx'))